# 🧠 Project 2: Production Chain-of-Thought Valuation Engine (Qwen2.5-3B)

### 📌 Overview & Purpose
This notebook implements an enterprise-ready valuation engine. Instead of predicting raw scalars directly, the model generates an XML-structured Chain-of-Thought reasoning block (`<thought>...</thought>`) that analyzes brand hints, product features, and price tiers before outputting the final price.

### ⚙️ Architecture & Optimizations
* **Base Model:** `unsloth/Qwen2.5-3B-Instruct`
* **Fine-Tuning Acceleration:** `Unsloth` custom Triton kernels with full Attention + MLP LoRA (`r=16`, `alpha=16`)
* **Data Efficiency:** Sequence Packing (`packing=True`, `max_seq_length=1024`) with `train_on_responses_only` mask
* **MLOps Integration:** Automated Weights & Biases telemetry logging
* **Edge Deployment:** Native 4-bit `Q4_K_M` GGUF quantization and direct Hugging Face Hub export for containerized Docker/Ollama inference

### 🔬 Observability & Production Utility
By enforcing explicit reasoning, the model allows failure root-cause audits and exposes intermediate outputs (e.g., market tier) for downstream agentic routing.

----------------
----------------

# Fine-Tuning unsloth/Qwen2.5-3B-Instruct

### Step 1: Environment Setup & Installations

In [1]:
# Install unsloth and required dependencies for efficient fine-tuning
!pip install -qq --no-cache-dir unsloth unsloth_zoo
!pip install -qq --no-deps torch peft accelerate bitsandbytes
!pip install -qq datasets sentencepiece protobuf evaluate wandb
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 263.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 342.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 270.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 233.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 152.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 307.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 253.1 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 353.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 266.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 349.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 207.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

## Architectural and Strategy 

1. Why SFT with **Chain-of-Thought(CoT)** over Raw **GRPO/Basic SFT**?
    * Why not basic SFT?

      Direct price guessing forces the model to predict numerical tokens in very first layer/token projection, causing heavy variance and hallucinations. CoT allocates compute budget at inference time to evaluate product attributes before settling on a value.

   * Why not GRPO(Group Relative Policy Optimization) in this specific setup?
  
     On a free T4 GPU(16GB VRAM), GRPO requires sampling N=4 completion candidates per prompt, making it *~4x to 6x slower* and highly prone to *Out-of-Memory(OOM)* crashes during rollouts. SFT with CoT achieves lower MAE(mean absolute error) within Kaggle's strict execution time limits.
  

2. Why **Outlier Data Cleaning**($0.5 to $2,000)?

    Raw web datasets contain extreme outliers($0.01 promotional items, corrupted inputs, or $50,000 extreme anomalies). Because squared/linear loss penalties scale drastically with magnitude, a single $10,000 outlier distorts gradient updates across an entire batch. Truncating these guarantees stable gradient steps.

3. Why `Qwen2.5-3B-Instruct` over Other Base Models?

   **Qwen2.5-3B-Instruct** possesses superior pre-trained mathematical reasoning and numerical tokenization compared to older models of similar size. It fits comfortably inside T4 GPU memory when paired with Unsloth's 4-bit quantization while leaving enough VRAM for sequence lengths of 1024.

4. Why Cosine Learning Rate Schedule & Targeted LoRA Projection?

   Regression and numerical fine-tuning require smooth parameter settling near convergence. A **Cosine LR Scheduler** systematically tapers off step-sizes to avoid overshooting target price distributions.

   Applying LoRA across **all projection layers**(`q, k, v, o, gate, up, down`) ensures the model adapts oth its feature extraction(attention) and feature mapping(MLP) components simultaneously.


----

GRPO is a Reinforcement learning technique used for fine-tuning LLMs to enhance complex reasoning and multi-step problem-solving cappabilities. Developed notably in **DeepSeek-R1** training pipeline, GRPO serves as a more efficient alternative to traditional methods like *Proximal Policy Optimization(PPO)* by eliminating the need for a separate "critic" value model.

The algorithm operates by generating multiple candidate outputs(rollouts) for each input and calculating rewards based on verifiable outcomes or custom reward functions. It then computes group-relative advantages by normalizing those rewards within the group, allowing the model to reinforce above-average responses while penalizing below-average ones. This approach enables stable, sample-efficient training without requiring extensive human-labeled preference data, making it particularly effective for tasks like mathematics, coding, and structured data extraction.


#### Reinforcement Fine-Tuning vs GRPO(Group Relative Policy Optimization)

RFT is a broad paradigm or framework if using RL to FT models, whereas GRPO is a specific algorithm used to implement that paradigm.
Like the difference between "Transportation"(RFT) and "Driving a Toyota"(GRPO). One is the general concept; the other is a specific, efficient engine designed for that concept.


* **Standard RFT(using PPO):** Requires training and running two models simultaneously: The **Policy Model** (the student) and a separate **Critic Model** (a value estimator that predicts how good a state is). This doubles memory usage and compute cost.
* **Efficient RFT(using GRPO):** Eliminates the Critic Model entirely. Instead of asking a separate model "how good is this?", GRPO generate a *group of responses* from the Policy Model itself and calculates the advantage by comparing them against group's average.

---

In [12]:
import os 
import re
import math 
import numpy as np
import pandas as pd
import torch
import wandb

from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel, is_bfloat16_supported
from huggingface_hub import login

In [5]:
# Configure W&B for lightweight tracking (logs metrics, but doesn't upload giant model checkpoints)
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_PROJECT"] = "ecommerce-price-cot-finetuning"

# Authenticate with HuggingFace & W&B 
login(token=os.environ.get("HF_TOKEN"))

In [6]:
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: subhrajyoti_soumyadarsan (subhrajyoti_soumyadarsan-independent-developer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Step 2: Model Initialization & Tokenizer Setup (Unsloth)

In [14]:
max_seq_length = 1024 # Expanded sequence length to comfortably fit <thought> reasoning tokens
load_in_4bit = True

# Load Qwen2.5-3B-Instruct with 4-bit quantization
# device_map = "balanced" automatically splits across Kaggle's Dual T4 GPUs if selected
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
    # device_map = "balanced" if torch.cuda.device_count() > 1 else "auto"
    device_map = {"": 0}
)

# Apply PEFT LoRA Adapters across attention and MLP layers
model = FastLanguageModel.get_peft_model(
    model, 
    r = 16, #rank like (4096x16 instead of 4096x4096)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,#scaling factor(α/r): controls the magnitude of change of adapter contribution wrt frozen base model weights
    lora_dropout = 0, # Disables dropout to use Unsloth's optimized fast Triton kernels
    bias = "none", #keeps bias terms frozen
    use_gradient_checkpointing = "unsloth", #activates Unsloth's memory efficient gradient checkpointing
    random_state = 3407,
)

==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

### Step 3: Data Preparation & Chain-of-Thought Preprocessing

In [8]:
SYSTEM_PROMPT = (
    "You are an expert e-commerce valuation system.\n"
    "First, analyze the item tier, category, and feature quality inside <thought>...</thought> tags.\n"
    "Then, output the final estimated retail value using the exact format 'Price: $XX.XX'."
)

def create_cot_reasoning(title, category, summary, price):
    """Constructs structured step-by-step reasoning blocks outputting target price."""
    val = float(price)

    # Stratify the pricing to give the model a logical starting bound
    if val < 20:
        tier = "Budget / Mass-marker tier"
    elif val < 100:
        tier = "Mid-range consumer product"
    elif val < 300:
        tier = "Premium / High-quality tier"
    else:
        tier = "High-end / Luxury segment"

    clean_title = title.strip() #strips surrounding whitespaces from the title
    
    brand_hint = clean_title.split()[0] if clean_title else "Generic" #extracts first word of the title as a proxy brand identifier

    formatted_price = f"${val:.2f}"

    cot_block = (
        f"<thought>\n"
        f"1. Category & Brand: Classifying within '{category}' (Brand hint: {brand_hint}).\n"
        f"2. Core Features: Evaluated key attributes: \"{summary[:90]}...\".\n"
        f"3. Market Tier: Assigned to {tier}.\n"
        f"4. Estimation Range: Value aligns with expected price distribution for {tier}.\n"
        f"</thought>\n"
        f"Price: {formatted_price}"
    )

    return cot_block

In [9]:
# Load items_lite dataset(22k)
raw_dataset = load_dataset("ed-donner/items_lite", split="train")

# Clean outliers ($0.50 to $2000.00) to stabilize regression gradient steps
def clean_data(example):
    try:
        p = float(example["price"])
        return 0.50 <= p <= 2000.00
    except:
        return False

filtered_dataset = raw_dataset.filter(clean_data)
split_dataset = filtered_dataset.train_test_split(test_size = 0.1, seed = 42)
train_raw = split_dataset["train"]
test_raw = split_dataset["test"]

def format_chatml_cot(examples):
    """Maps the dataset into ChatML conversational format."""
    texts = []
    for title, cat, summary, price in zip(examples["title"], examples["category"], examples["summary"], examples["price"]):
        user_content = f"Product Title: {title}\nCategory: {cat}\nSummary: {summary}"
        assistant_content = create_cot_reasoning(title, cat, summary, price)

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content}
        ]

        # Apply Qwen's specific ChatML template
        rendered_text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
        texts.append(rendered_text)
        
    return {"text": texts} #map(batched=True) requires a dictionary with the mapped column

train_dataset = train_raw.map(format_chatml_cot, batched=True) #applis formatting across the training split in batches

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

### Step 4: Fine-Tuning(SFT)

In [15]:
# Initialize Weights & Biases run
wandb.init(project="ecommerce-price-cot-finetuning", name="qwen2.5-3b-cot-run-1")

# version update 
# trainer = SFTTrainer(
#     model = model,
#     processing_class = tokenizer,
#     train_dataset = train_dataset,
#     # dataset_text_field = "text", #tells the trainer which column in HF dataset dictionary contains the pre-formatted text strings to tokenize
#     max_seq_length = max_seq_length, 
#     dataset_num_proc = 2, #spawns 2 parallel CPU worker processes to speed up data preprocessing and tokenization during dataset preparation
#     packing = False, #when false each sample padded individually to form batches, when true concatenated till max_seq_len to maximize GPU efficiency
#     args = TrainingArguments(
#         per_device_train_batch_size = 2,#no.of training samples processed simultaneously on a GPU in 1 forward/backward pass. Kept low to fit VRAM
#         gradient_accumulation_steps = 4, #accumulates gradients over 4 fwd/bwd passes before calling optimizer.step() to update model weights
#         # Effective batch_size=batchsize_per_device x accu_steps x no_gpu = 2x4x1=8. This simulates batchsize of 8 without 8x the VRAM.
#         warmup_steps = 15, #warming up lr to prevent unstable, disruptive gradient updates at start of training.
#         max_steps = 150, #total no.of optimizer steps to train for. traininf stops regardless of how many epochs completed.
#         learning_rate = 2e-4,
#         fp16 = not is_bfloat16_supported(),
#         bf16 = is_bfloat16_supported(),
#         logging_steps = 10, #logs loss, lr, and step time every 10 update steps
#         optim = "adamw_8bit", #bitsandbyrtes 8-bit quantized AdamW optimizer.
#         weight_decay = 0.01, # L2 regularization penalty added to loss function to shrink weights and prevent overfitting.
#         lr_scheduler_type = "cosine", #lr after warmup, decays following cosine curve towards 0 in 150 steps, allowing fine-grained convergence
#         seed = 3407,
#         output_dir = "/kaggle/working/outputs", # Write directly to Kaggle working directory
#         report_to = "wandb"
#     ),
# )

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer, # Note: if you still get a warning, change this to processing_class = tokenizer
    train_dataset = train_dataset,
    # All trainer arguments and SFT-specific arguments are now merged into SFTConfig
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,
        max_steps = 150,
        # num_train_epochs = 1.0,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/kaggle/working/outputs",
        report_to = "wandb"
    ),
)

print("\nStarting Fine-Tuning on Kaggle ---")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.

Starting Fine-Tuning on Kaggle ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,000 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,3.137969
20,1.964733
30,1.015707
40,0.921461
50,0.829790
60,0.806338
70,0.854780
80,0.858357
90,0.829991
100,0.843827


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-150/tokenizer_config.json.


TrainOutput(global_step=150, training_loss=1.0812423642476399, metrics={'train_runtime': 586.9192, 'train_samples_per_second': 2.045, 'train_steps_per_second': 0.256, 'total_flos': 6283228763308032.0, 'train_loss': 1.0812423642476399, 'epoch': 0.06666666666666667})

### Step 5: Inference Evaluation & Regression Metrics

In [16]:
FastLanguageModel.for_inference(model)

def parse_price(text: str) -> float | None:
    """Robust regex parser that strips <thought> block and extracts price string."""
    target_str = text.split("</thought>")[-1] if "</thought>" in text else text

    # First attempt: Look for 'Price: $XX.XX'
    match = re.search(r"Price:\s*\$\s*(\d+(?:\.\d{1,2})?)", target_str, re.IGNORECASE)
    if not match:
        # Fallback: Look for any isolated dollar amount
        match = re.search(r"\$\s*(\d+(?:\.\d{1,2})?)", target_str)

    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return None
    return None

actual_prices = []
predicted_prices = []

# Testing on a small subset for rapid validation
eval_samples = min(50, len(test_raw))
print(f"\nRunning Evaluation on {eval_samples} Test Items ---")

for i in range(eval_samples):
    item = test_raw[i]
    true_price = float(item["price"])

    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Product Title: {item['title']}\nCategory: {item['category']}\nSummary: {item['summary']}"}
    ]

    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate( #generates upto 160 new tokens with a low temperature of 0.1 for deterministic outputs.
        input_ids = inputs,
        max_new_tokens = 160, #ample space for CoT reasoning + price
        temperature = 0.1,
        use_cache = True
    )

    decoded = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens = True)
    pred_price = parse_price(decoded)

    if pred_price is not None:
        actual_prices.append(true_price)
        predicted_prices.append(pred_price)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Running Evaluation on 50 Test Items ---


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [17]:
# Metrics Calculations(Final Regression Metrics)
y_true = np.array(actual_prices)
y_pred = np.array(predicted_prices)

mae = np.mean(np.abs(y_true - y_pred))
mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-5))) * 100
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
log_error = np.mean(np.abs(np.log1p(y_true) - np.log1p(y_pred)))

print(f"\n================ FINAL EVALUATION METRICS ================")
print(f"Successfully Evaluated : {len(y_true)} / {eval_samples} samples")
print(f"Mean Absolute Error (MAE)      : ${mae:.2f}")
print(f"Mean Absolute % Error (MAPE)   : {mape:.2f}%")
print(f"Root Mean Squared Error (RMSE) : ${rmse:.2f}")
print(f"Logarithmic Error (RMSLE)      : {log_error:.4f}")
print("==================================================================")


================ FINAL EVALUATION METRICS ================
Successfully Evaluated : 50 / 50 samples
Mean Absolute Error (MAE)      : $109.61
Mean Absolute % Error (MAPE)   : 57.51%
Root Mean Squared Error (RMSE) : $216.85
Logarithmic Error (RMSLE)      : 0.6707


In [18]:
# Log final evaluation to W&B
wandb.log({"eval/mae": mae, "eval/mape": mape, "eval/rmse": rmse, "eval/rmsle": log_error})
wandb.finish()

eval/mae,▁
eval/mape,▁
eval/rmse,▁
eval/rmsle,▁
train/epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
train/global_step,▁▁▂▃▃▃▄▅▅▅▆▇▇▇███
train/grad_norm,█▅▂▂▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▅██▇▇▆▆▅▄▃▂▂▁▁▁
train/loss,█▄▂▁▁▁▁▁▁▁▁▁▁▁▁
eval/mae,109.614
eval/mape,57.50736


### 6. Export and Push to HuggingFace

In [19]:
HF_USERNAME = "Subhrajyoti75"
REPO_ID = f"{HF_USERNAME}/qwen2.5-3b-cot-pricing-gguf"

print("\nSaving LoRA adapters and exporting to GGUF (Q4_K_M)...")

# Push raw LoRA adapters 
model.push_to_hub(f"{REPO_ID}-lora")

# Merge LoRA weights into 4-bit GGUF and push directly to HF hub
# This enables easy downloading for Ollama Docker serving later
model.push_to_hub_gguf(
    REPO_ID,
    tokenizer,
    quantization_method="q4_k_m",
    token=os.environ.get("HF_TOKEN")
)



Saving LoRA adapters and exporting to GGUF (Q4_K_M)...


README.md:   0%|          | 0.00/583 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Subhrajyoti75/qwen2.5-3b-cot-pricing-gguf-lora
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_w0rmo3mx`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10472-mix-4b653db (app-b10472-mix-4b653db-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_w0rmo3mx_gguf/qwen2.5-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversio

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Subhrajyoti75/qwen2.5-3b-cot-pricing-gguf
Unsloth: Cleaning up temporary files...


'Subhrajyoti75/qwen2.5-3b-cot-pricing-gguf'

In [21]:
# ====================================================================OPTIONAL=============================================================
# Util function from Project 1
from util import evaluate

# 1. Adapt test_raw with the 'completion' and 'prompt' keys expected by util.py
test_eval = test_raw.map(lambda x: {
    "completion": str(x["price"]),
    "prompt": f"Title: {x['title']}\nCategory: {x['category']}\nSummary: {x['summary']}"
})

# 2. Updated prediction function for Chain-of-Thought (CoT) model
def model_predict(item):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Product Title: {item['title']}\nCategory: {item['category']}\nSummary: {item['summary']}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs, 
            max_new_tokens=160, # Ample room for <thought>...</thought> reasoning
            temperature=0.1,
            use_cache=True
        )
        
    prompt_len = inputs.shape[1]
    generated_ids = output_ids[0, prompt_len:]
    decoded_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Extract the scalar value using the CoT regex parser
    parsed_price = parse_price(decoded_text)
    
    if parsed_price is not None:
        return f"${parsed_price:.2f}"
    else:
        return "$0.00"

# 3. Run evaluation (set size to evaluate e.g. 50 or 100 items; default is 200)
evaluate(model_predict, test_eval, size=50)

## Developing results further

In [22]:
# Initialize Weights & Biases run
wandb.init(project="ecommerce-price-cot-finetuning", name="qwen2.5-3b-cot-run-2")
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer, # Note: if you still get a warning, change this to processing_class = tokenizer
    train_dataset = train_dataset,
    # All trainer arguments and SFT-specific arguments are now merged into SFTConfig
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = True,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 25,
        max_steps = 500,
        # num_train_epochs = 1.0,
        learning_rate = 3e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/kaggle/working/outputs",
        report_to = "wandb"
    ),
)

# OPTIMIZATION: Force the model to only calculate loss on the CoT and Price
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

print("\nStarting Fine-Tuning on Kaggle ---")
trainer.train()

[unsloth.trainer|WARNING]Unsloth: packing=True ignored (UNSLOTH_RETURN_LOGITS=1).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/18000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/18000 [00:00<?, ? examples/s]


Starting Fine-Tuning on Kaggle ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.124230
20,0.132047
30,0.122040
40,0.122060
50,0.115851
60,0.115211
70,0.112615
80,0.106547
90,0.110852
100,0.116796


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-500/tokenizer_config.json.


TrainOutput(global_step=500, training_loss=0.09917598700523376, metrics={'train_runtime': 2000.9335, 'train_samples_per_second': 1.999, 'train_steps_per_second': 0.25, 'total_flos': 2.210278778786611e+16, 'train_loss': 0.09917598700523376, 'epoch': 0.2222222222222222})

In [23]:
from tqdm.auto import tqdm
FastLanguageModel.for_inference(model)

def parse_price(text: str) -> float | None:
    """Robust regex parser that strips <thought> block and extracts price string."""
    target_str = text.split("</thought>")[-1] if "</thought>" in text else text

    # First attempt: Look for 'Price: $XX.XX'
    match = re.search(r"Price:\s*\$\s*(\d+(?:\.\d{1,2})?)", target_str, re.IGNORECASE)
    if not match:
        # Fallback: Look for any isolated dollar amount
        match = re.search(r"\$\s*(\d+(?:\.\d{1,2})?)", target_str)

    if match:
        try:
            return float(match.group(1))
        except ValueError:
            return None
    return None

actual_prices = []
predicted_prices = []

# Testing on a small subset for rapid validation
eval_samples = min(50, len(test_raw))
print(f"\nRunning Evaluation on {eval_samples} Test Items ---")

for i in tqdm(range(eval_samples)):
    item = test_raw[i]
    true_price = float(item["price"])

    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Product Title: {item['title']}\nCategory: {item['category']}\nSummary: {item['summary']}"}
    ]

    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt"
    ).to("cuda")

    outputs = model.generate( #generates upto 160 new tokens with a low temperature of 0.1 for deterministic outputs.
        input_ids = inputs,
        max_new_tokens = 160, #ample space for CoT reasoning + price
        temperature = 0.1,
        use_cache = True
    )

    decoded = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens = True)
    pred_price = parse_price(decoded)

    if pred_price is not None:
        actual_prices.append(true_price)
        predicted_prices.append(pred_price)



Running Evaluation on 50 Test Items ---


  0%|          | 0/50 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

In [24]:
# Metrics Calculations(Final Regression Metrics)
y_true = np.array(actual_prices)
y_pred = np.array(predicted_prices)

mae = np.mean(np.abs(y_true - y_pred))
mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1e-5))) * 100
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
log_error = np.mean(np.abs(np.log1p(y_true) - np.log1p(y_pred)))

print(f"\n================ FINAL EVALUATION METRICS ================")
print(f"Successfully Evaluated : {len(y_true)} / {eval_samples} samples")
print(f"Mean Absolute Error (MAE)      : ${mae:.2f}")
print(f"Mean Absolute % Error (MAPE)   : {mape:.2f}%")
print(f"Root Mean Squared Error (RMSE) : ${rmse:.2f}")
print(f"Logarithmic Error (RMSLE)      : {log_error:.4f}")
print("==================================================================")


================ FINAL EVALUATION METRICS ================
Successfully Evaluated : 50 / 50 samples
Mean Absolute Error (MAE)      : $94.28
Mean Absolute % Error (MAPE)   : 65.64%
Root Mean Squared Error (RMSE) : $195.00
Logarithmic Error (RMSLE)      : 0.5431


In [25]:
HF_USERNAME = "Subhrajyoti75"
REPO_ID = f"{HF_USERNAME}/qwen2.5-3b-cot-pricing-gguf_2"

print("\nSaving LoRA adapters and exporting to GGUF (Q4_K_M)...")

# Push raw LoRA adapters 
model.push_to_hub(f"{REPO_ID}-lora")

# Merge LoRA weights into 4-bit GGUF and push directly to HF hub
# This enables easy downloading for Ollama Docker serving later
model.push_to_hub_gguf(
    REPO_ID,
    tokenizer,
    quantization_method="q4_k_m",
    token=os.environ.get("HF_TOKEN")
)


Saving LoRA adapters and exporting to GGUF (Q4_K_M)...


README.md:   0%|          | 0.00/583 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Subhrajyoti75/qwen2.5-3b-cot-pricing-gguf_2-lora
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_j9_dhihv`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_j9_dhihv_gguf/qwen2.5-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_j9_dhihv_gguf/qwen2.5-3b-instruct.Q4_K_M.gguf']
Unsloth: exam

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Subhrajyoti75/qwen2.5-3b-cot-pricing-gguf_2
Unsloth: Cleaning up temporary files...


'Subhrajyoti75/qwen2.5-3b-cot-pricing-gguf_2'

In [26]:
# ====================================================================OPTIONAL=============================================================
# Util function from Project 1
from util import evaluate

# 1. Adapt test_raw with the 'completion' and 'prompt' keys expected by util.py
test_eval = test_raw.map(lambda x: {
    "completion": str(x["price"]),
    "prompt": f"Title: {x['title']}\nCategory: {x['category']}\nSummary: {x['summary']}"
})

# 2. Updated prediction function for Chain-of-Thought (CoT) model
def model_predict(item):
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Product Title: {item['title']}\nCategory: {item['category']}\nSummary: {item['summary']}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs, 
            max_new_tokens=160, # Ample room for <thought>...</thought> reasoning
            temperature=0.1,
            use_cache=True
        )
        
    prompt_len = inputs.shape[1]
    generated_ids = output_ids[0, prompt_len:]
    decoded_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Extract the scalar value using the CoT regex parser
    parsed_price = parse_price(decoded_text)
    
    if parsed_price is not None:
        return f"${parsed_price:.2f}"
    else:
        return "$0.00"

# 3. Run evaluation (set size to evaluate e.g. 50 or 100 items; default is 200)
evaluate(model_predict, test_eval, size=50)

In [ ]:
# Run this to get model output directly which is deployed locally as a REST API using docker desktop
import requests

url = "http://localhost:11434/api/generate"
payload = {
    "model": "cot-pricer",
    "prompt": "Product Title: Anker USB-C Cable\nCategory: Electronics\nSummary: A durable 6-foot braided charging cable.",
    "stream": False
}

response = requests.post(url, json=payload)
print(response.json()["response"])